# Real-Time VAD with faster-whisper

This notebook integrates WebRTC VAD with faster-whisper for real-time speech recognition.

## What this implements
- Continuous audio capture with sounddevice
- Real-time voice activity detection
- Automatic speech segmentation
- Faster-whisper transcription

## Architecture
```
Mic → VAD → [Speech Detected] → Capture → Whisper → Text
```

## This evolved into the production pipeline in Yumi_Hears/

## Dependencies
```bash
pip install sounddevice webrtcvad-wheels faster-whisper numpy scipy
```

In [ ]:
# Microphone
# ↓
# Real-time audio stream
# ↓
# float32 → int16
# ↓
# WebRTC VAD
# ↓
# speech start detection
# ↓
# speech end detection
# ↓
# saved speech file

In [ ]:
# sounddevice webrtcvad numpy scipy

In [ ]:
import sounddevice as sd
import numpy as np
import webrtcvad
from scipy.io.wavfile import write

In [ ]:
# CONFIG
# =====================

SAMPLE_RATE = 16000
FRAME_DURATION = 20  # ms
FRAME_SIZE = int(SAMPLE_RATE * FRAME_DURATION / 1000)

VAD_MODE = 2

vad = webrtcvad.Vad(VAD_MODE)

In [ ]:
# =====================
# FLOAT → INT16
# =====================

def float_to_int16(audio):

    audio = np.clip(audio, -1, 1)

    return (audio * 32767).astype(np.int16)


In [ ]:
# =====================
# MAIN LISTENER
# =====================

def listen_for_speech():

    print("Listening...")

    stream = sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=FRAME_SIZE
    )

    stream.start()

    recording = []
    speech_detected = False
    silence_counter = 0

    while True:

        frame, _ = stream.read(FRAME_SIZE)

        frame = frame.flatten()

        frame_int16 = float_to_int16(frame)

        is_speech = vad.is_speech(frame_int16.tobytes(), SAMPLE_RATE)

        if is_speech:

            if not speech_detected:
                print("Speech started")

            speech_detected = True
            silence_counter = 0

            recording.extend(frame_int16)

        elif speech_detected:

            silence_counter += 1

            recording.extend(frame_int16)

            # stop after ~0.8 seconds of silence
            if silence_counter > 40:
                print("Speech ended")

                break

    stream.stop()

    audio = np.array(recording, dtype=np.int16)

    write("detected_speech.wav", SAMPLE_RATE, audio)

    print("Saved detected_speech.wav")

    return "detected_speech.wav"

In [ ]:
listen_for_speech()

In [ ]:
from IPython.display import Audio

Audio("detected_speech.wav")